In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("DATA_SET.csv")

In [ ]:
null_pct = (df.isnull().sum() / len(df)) * 100
null_pct = null_pct[null_pct > 0]
null_pct = null_pct.sort_values(ascending=False)
print(null_pct)

In [ ]:
#dropping the columns that are null more then 80 percent
df.drop(columns=['PoolQC','MiscFeature','Alley','Fence'],axis=1,inplace=True)

In [ ]:
import pandas as pd

# Show all columns
pd.set_option('display.max_columns', None)
df.head(11)


In [ ]:
df['MSSubClass'] = df['MSSubClass'].astype(str)
df['MoSold'] = df['MoSold'].astype(str)

In [ ]:
# Categorical — fill with 'None' (feature doesn't exist)
cat_null_cols = ['MasVnrType', 'FireplaceQu', 'GarageType', 'GarageFinish',
                 'GarageQual', 'GarageCond', 'BsmtExposure', 'BsmtFinType2',
                 'BsmtQual', 'BsmtCond', 'BsmtFinType1']

for col in cat_null_cols:
    df[col].fillna('None', inplace=True)

# Electrical — only 1 missing, fill with mode (truly random)
df['Electrical'].fillna(df['Electrical'].mode()[0], inplace=True)

# Numerical — median (we just don't know the value)
df['LotFrontage'].fillna(df['LotFrontage'].median(), inplace=True)

# Numerical — 0 (feature doesn't exist)
df['GarageYrBlt'].fillna(0, inplace=True)
df['MasVnrArea'].fillna(0, inplace=True)

In [ ]:
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'GarageQual', 'GarageCond', 
                'FireplaceQu']

for col in ordinal_cols:
    df[col] = df[col].map(quality_map)

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
for col in df.select_dtypes('bool').columns:
    df[col] = df[col].astype(int)

In [ ]:
df.shape

In [ ]:
print(df.isnull().sum().sum())   # should be 0
print(df.dtypes.value_counts())  # should be int64/float64 only, no bool no object

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = df.drop('SalePrice', axis=1)
y = np.log1p(df['SalePrice'])  # 1. transform first

X_train, X_test, y_train, y_test = train_test_split(  # 2. then split
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

alphas = [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]
model = RidgeCV(alphas=alphas, cv=5)
model.fit(X_train, y_train)

print(f'Best Alpha: {model.alpha_}')
y_pred = model.predict(X_test)
print(f"R2 Score:  {r2_score(y_test, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

In [ ]:
import joblib

joblib.dump(model, '../backend/ridge_model.pkl')
joblib.dump(scaler, '../backend/scaler.pkl')
joblib.dump(list(df.drop('SalePrice', axis=1).columns), '../backend/columns.pkl')

print("saved!")